# Prediction

Use existing databases to make predictions and perform interpolation.

In [1]:
import pprint

import cutcutcodec

from mendevi.models.nn import DNN
from mendevi.models.gpr import GPR
from mendevi.models.lr import LR

In [2]:
dataset = "svtav1_vs_rav1e_vs_aom.db"  # the dataset name, automatically downloaded
val = 0.2  # keep 80% of data for training, 20% for validation
video = cutcutcodec.utils.get_project_root() / "media" / "video" / "intro.webm"

## Biaised linear regression

Predicts the output as a linear combination of the inputs, plus a constant.
It is a very fast model $O(n)$.

In [3]:
# model training
lr_model = LR(
    input_labels=["effort", "encoder", "pixels", "quality", "rms_sobel", "rms_time_diff"],
    aggregation=["encoder"],  # optional, this feature provides one model per encoder
    output_labels=["log_rate", "psnr", "ssim"],
).fit(
    dataset,
    table="t_enc_encode",  # optional, help to find the good SQL request
    select=f"uniform(vid_hash) >= {val}",
)

Fit 'Linear Regression for a predictive model'...
█ get SQL extractor
█ perform SQL query
█ Fit on cluster encoder=libaom-av1...
█ █ this cluster contains 2428 observations
█ █ 152.72 ms
█ Fit on cluster encoder=librav1e...
█ █ this cluster contains 2473 observations
█ █ 143.86 ms
█ Fit on cluster encoder=libsvtav1...
█ █ this cluster contains 2472 observations
█ █ 153.85 ms
█ 8.79 s


In [4]:
# validation from dataset
lr_model.validate(dataset, table="t_enc_encode", select=f"uniform(vid_hash) < {val}")
# print(lr_model.accuracy)  # for each cluster name, associate for each output label, the predicted and validation data

Validate 'Linear Regression for a predictive model'...
█ get SQL extractor
█ perform SQL query
█ Validate on cluster encoder=libaom-av1...
█ █ this cluster contains 596 observations
█ █ 55.79 ms
█ Validate on cluster encoder=librav1e...
█ █ this cluster contains 615 observations
█ █ 47.68 ms
█ Validate on cluster encoder=libsvtav1...
█ █ this cluster contains 613 observations
█ █ 48.87 ms
█ 4.49 s
MAPE of log_rate: 3.54%
RMS  of log_rate: 0.2951
MAPE of psnr: 11.65%
RMS  of psnr: 6.709
MAPE of ssim: 4.17%
RMS  of ssim: 0.05084
MAPE of rate: 66.81%
RMS  of rate: 1.593e+07


In [5]:
# prediction from a video
pred = lr_model.predict_from_video(video, effort="medium", encoder="libsvtav1", quality=0.5)
pprint.pprint(pred)

Predict 'Linear Regression for a predictive model'...
█ check args
█ vectorize args
█ predict
█ Predict for cluster encoder=libsvtav1...
█ █ this cluster contains 1 observations
█ █ 10.24 ms
█ 45.09 ms


{'log_rate': [6.372843265533447],
 'psnr': [34.864139556884766],
 'ssim': [0.9203274250030518]}


## Gaussian process regression

Very slow model $O(n^3)$. Efficient when data are very sparse.

In [6]:
# model training
gpr_model = GPR(  # no aggregation because it fits only on continuous parameters
    input_labels=["effort", "encoder", "pixels", "quality", "rms_sobel", "rms_time_diff"],
    output_labels=["log_rate", "psnr", "ssim"],
).fit(
    dataset,
    table="t_enc_encode",  # optional, help to find the good SQL request
    select=f"uniform(vid_hash) >= {val}",
)

Fit 'Gaussian Process Regression predictive model'...
█ get SQL extractor
█ perform SQL query
█ Fit on cluster effort=medium, encoder=libaom-av1...
█ █ this cluster contains 804 observations
/home/rrichard/.pyenv/versions/3.15.0rc2/envs/3.15/lib/python3.15/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
█ █ 4 min 31 s
█ Fit on cluster effort=slow, encoder=libaom-av1...
█ █ this cluster contains 811 observations
/home/rrichard/.pyenv/versions/3.15.0rc2/envs/3.15/lib/python3.15/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
█ █ 6 min 25 s
█ Fi

In [7]:
# validation from dataset
gpr_model.validate(dataset, table="t_enc_encode", select=f"uniform(vid_hash) < {val}")

Validate 'Gaussian Process Regression predictive model'...
█ get SQL extractor
█ perform SQL query
█ Validate on cluster effort=fast, encoder=libaom-av1...
█ █ this cluster contains 201 observations
█ █ 25.72 ms
█ Validate on cluster effort=fast, encoder=librav1e...
█ █ this cluster contains 181 observations
█ █ 32.45 ms
█ Validate on cluster effort=slow, encoder=librav1e...
█ █ this cluster contains 208 observations
█ █ 37.51 ms
█ Validate on cluster effort=medium, encoder=libsvtav1...
█ █ this cluster contains 196 observations
█ █ 40.94 ms
█ Validate on cluster effort=medium, encoder=libaom-av1...
█ █ this cluster contains 202 observations
█ █ 30.09 ms
█ Validate on cluster effort=medium, encoder=librav1e...
█ █ this cluster contains 226 observations
█ █ 35.29 ms
█ Validate on cluster effort=fast, encoder=libsvtav1...
█ █ this cluster contains 212 observations
█ █ 31.69 ms
█ Validate on cluster effort=slow, encoder=libsvtav1...
█ █ this cluster contains 205 observations
█ █ 30.08 ms


In [8]:
# prediction from a video
pred = gpr_model.predict_from_video(video, effort="medium", encoder="libsvtav1", quality=0.5)
pprint.pprint(pred)

Predict 'Gaussian Process Regression predictive model'...
█ check args
█ vectorize args
█ predict
█ Predict for cluster effort=medium, encoder=libsvtav1...
█ █ this cluster contains 1 observations
█ █ 12.82 ms
█ 40.70 ms


{'log_rate': [6.327295780181885],
 'psnr': [40.00262451171875],
 'ssim': [0.9444530606269836]}


## Deep neuronal network

Medium speed model, $O(n)$.

In [9]:
# model training
dnn_model = DNN(  # see parameters niter, learning_rate and total
    input_labels=["effort", "encoder", "pixels", "quality", "rms_sobel", "rms_time_diff"],
    aggregation=[],  # optional, this feature force a single common model
    output_labels=["log_rate", "psnr", "ssim"],
).fit(
    dataset,
    table="t_enc_encode",  # optional, help to find the good SQL request
    select=f"uniform(vid_hash) >= {val}",
)

Fit 'regressive model to predict log_rate, psnr, ssim from effort, encoder, pixels, quality, rms_sobel, rms_time_diff'...
█ get SQL extractor
█ perform SQL query
█ Fit on cluster ...
█ █ this cluster contains 7373 observations
█ █ 3 min 56 s                                                                                                       
█ 4 min 4 s


In [10]:
# validation from dataset
dnn_model.validate(dataset, table="t_enc_encode", select=f"uniform(vid_hash) < {val}")

Validate 'regressive model to predict log_rate, psnr, ssim from effort, encoder, pixels, quality, rms_sobel, rms_time_diff'...
█ get SQL extractor
█ perform SQL query
█ Validate on cluster ...
█ █ this cluster contains 1824 observations
█ █ 90.94 ms
█ 3.92 s
MAPE of log_rate: 2.81%
RMS  of log_rate: 0.2255
MAPE of psnr: 6.84%
RMS  of psnr: 3.822
MAPE of ssim: 2.30%
RMS  of ssim: 0.03144
MAPE of rate: 44.32%
RMS  of rate: 1.363e+07


In [11]:
# prediction from a video
pred = dnn_model.predict_from_video(video, effort="medium", encoder="libsvtav1", quality=0.5)
pprint.pprint(pred)

Predict 'regressive model to predict log_rate, psnr, ssim from effort, encoder, pixels, quality, rms_sobel, rms_time_diff'...
█ check args
█ vectorize args
█ predict
█ Predict for cluster ...
█ █ this cluster contains 1 observations
█ █ 9.75 ms
█ 32.08 ms


{'log_rate': [5.780920505523682],
 'psnr': [42.27354049682617],
 'ssim': [0.9299823641777039]}
